# 07 - SharePoint indexed (C#): Content Understanding, embeddings and Entra ACLs

## Required permissions

### User permissions

| Who needs it | Task | Required permission or role | Scope / when needed |
|---|---|---|---|
| User running the Search setup cells | Create indexes, data sources, skillsets and indexers | **Search Service Contributor** | Azure AI Search service |
| Each signed-in user running queries | Query indexed documents | **Search Index Data Reader** (Data Contributor also works), plus access to the source documents | Search service and SharePoint documents |
| User configuring the app registration | Create/configure the ingestion app and its federated credential | Permission to register apps and manage this app, such as app ownership or **Application Administrator** | Microsoft Entra; one-time setup |
| Administrator granting tenant consent | Grant the app's API permissions admin consent | **Privileged Role Administrator** or **Global Administrator** | Microsoft Entra; includes Microsoft Graph application permissions |
| Administrator granting site access | Grant the app access to the test site using the delegated setup path | **SharePoint Administrator** or higher, site membership, and Graph delegated **`Sites.FullControl.All`** in the calling token | Target SharePoint site; one-time setup |

The final two tasks can be completed by separately authorized accounts. Azure subscription Owner does not replace Entra or SharePoint authorization. Once app setup is complete, running ingestion and queries needs only the first two rows. Section 2c also documents an explicit app-only alternative for the site grant.

### App registration permissions

Configure these on the **ingestion app**, not on the notebook user's account. All API permissions below are **Application** permissions and require admin consent.

| Who needs it | API / resource | Permission | Purpose |
|---|---|---|---|
| Ingestion app's service principal | Microsoft Graph | **`Files.Read.All`** | Read document-library files for ingestion; tenant-wide file read |
| Ingestion app's service principal | Microsoft Graph | **`Sites.Selected`** | Selected-site authorization for document ACL ingestion |
| Ingestion app's service principal | Target SharePoint site | Explicit **Read** grant to the app | Authorize the selected site separately from admin consent |

The app also needs a **federated credential trusting Search's system-assigned managed identity**; no client secret is used. `Sites.Selected` does not narrow the separate tenant-wide `Files.Read.All` grant. The setup user's `Sites.FullControl.All` is not a permanent runtime permission for this app.

| Who needs it | Permission | Scope / purpose |
|---|---|---|
| Search system-assigned managed identity | **Cognitive Services User** | Parent Foundry resource; keyless Content Understanding and text embeddings. Bicep already declares this grant. |

Sources: [Search ACL permission matrix](https://learn.microsoft.com/azure/search/search-indexer-sharepoint-access-control-lists#permissions-by-acl-scenario), [admin consent roles](https://learn.microsoft.com/entra/identity/enterprise-apps/grant-admin-consent#prerequisites), [site-grant permissions](https://learn.microsoft.com/graph/api/site-post-permissions#permissions).

**Resource prefix:** use `WORKSHOP_PREFIX`, for example `alice01` (3-20 lowercase letters/digits, starting with a letter). The index, data source, skillset and indexer use `alice01-nb7-*`, following the reference notebook's naming. Existing `alice01-sp-*` objects are left unchanged. Reuse the ingestion app configured by `scripts\setup-sharepoint-app.ps1`; this notebook does not create another app registration. Prefixes prevent naming collisions, not enforce access isolation.

This is the text-only adaptation of [01_setup_index.ipynb](https://github.com/memasanz/foundry-iq-sharepoint-indexer/blob/main/notebooks/01_setup_index.ipynb): a small REST helper, Content Understanding semantic chunks, text embeddings, page/source metadata and ACL-preserving projections. There are no Vision skills, image vectorizers, normalized-image projections or image-description model calls. The same content can be queried without an index in **notebook 08 (remote)**.

| What runs | Where content lives | App registration |
|---|---|---|
| SharePoint indexer -> Content Understanding -> text embeddings -> ACL-preserving projections -> Search index | Markdown chunks, text vectors and source/page metadata in Azure AI Search | Required for ingestion |

This lab demonstrates **Entra user and group ACLs**, not native SharePoint-group expansion. Give test users or Entra groups access directly to the test documents. Changing the extraction pipeline does not fix unsupported group nesting or make every document visible in SharePoint visible through Search; verify allowed and restricted users separately.

**Run order:** run section 1, complete the one-time app setup in section 2, then run sections 3-6. `scripts\setup-sharepoint-app.ps1` automates app configuration and the site grant. Reuse an already configured app rather than creating another. Cleanup is opt-in.

**C# option:** the [.NET 10 console app and detailed step-by-step README](../c-sharp-console-apps/07-sharepoint-indexed/README.md)
are supported alternatives to this optional legacy Polyglot/.NET Interactive notebook.
All BCL helpers are inline: no other notebook, console import, local SDK packages or Python runtime are required by the VS Code C# path.
Payloads and `<prefix>-nb7-*` names match Python07 and lab 11. Both languages target the same Search objects with the same prefix.

## Prerequisites and safe test content

- Use the workshop tenant and a dedicated **non-sensitive test SharePoint site**, not a production library. Search and SharePoint should be in the same tenant for this lab. Sign in with `az login --tenant <tenant-id>`.
- Complete the [SharePoint indexer preview registration](https://aka.ms/azure-cognitive-search/indexer-preview). Review current preview limitations, including Conditional Access incompatibilities, unsupported guest-user ACLs and unsupported Entra groups nested inside SharePoint groups. Do not disable tenant security policies to make a lab run.
- Use the user and app permission tables above. Admin consent, the selected-site grant and managed-identity federation are handled by the setup scripts before ingestion. This notebook reuses that configuration; it does not repeat Graph or ARM federation checks.
- Prepare two small, non-sensitive documents in the test site's default Documents library. Give one test user or an Entra group direct access to the restricted document; the second test user must not have another access path. Do not rely on an Entra group nested inside a SharePoint group, or on site/group ownership alone.
- Reuse the deployed Foundry resource and `workshop-embedding` deployment. Content Understanding needs a supported Foundry region, normally East US, and the Search identity's Cognitive Services User role. Analysis and embeddings are billable; analysis can time out after five minutes and still incur charges. This text-only pipeline needs no chat-model deployment.
- Add `SHAREPOINT_SITE_URL` and a unique `WORKSHOP_PREFIX` to your untracked `.env`. Set the registration identifiers after section 2. Never paste access tokens or client secrets into cells, output, or chat.

These are preview APIs. Review the linked documentation before running.

For this notebook, select the C# kernel and use **`c-sharp-notebooks` as the working directory**.
Root `.env` then `.env.azure` override environment settings, with `.env.azure` winning.
Set `AZURE_TENANT_ID`, `AZURE_SEARCH_ENDPOINT`, `SHAREPOINT_SITE_URL`, `AZURE_AI_ACCOUNT_NAME`,
`AZURE_AI_EMBEDDING_DEPLOYMENT_NAME` and `WORKSHOP_PREFIX`. Optional embedding model defaults to
`text-embedding-3-small`; dimensions remain 1536. After administrator setup, set the two client IDs described below.
Queries against an existing index need no ingestion app IDs. No Blob, chat model, Web IQ key or APIM is required.

## 1. Inline helpers and configuration

Run the next four C# cells in order: **1a imports, 1b settings helper, 1c REST helper,
then 1d configuration**. `SharePointSettings` and `SharePointTransport` are custom types:
their complete definitions must execute before configuration uses them. After a kernel restart,
rerun all four cells. The later cells contain the actual
JSON payloads and REST calls in Python07's order, rather than invoking a console workflow.
It reads configuration and validates public-cloud URLs/names, but makes **no token, network or file-write request**.
Tokens are acquired freshly for Search (`https://search.azure.com`) on later requests, and never persisted.
Simple quoted `KEY=VALUE`, comments and `export` are supported; no dotenv interpolation/multiline values.

The console saves a small run checkpoint for independent processes. This notebook deliberately keeps
`previous_run_start` **only in kernel memory** and writes no console cache. After a restart,
monitoring observes the current last result unless you explicitly request/capture another run.
HTTP failures show bounded, token-redacted diagnostics; no automatic paid retries.

### 1a. Import .NET namespaces

Run this cell first. Select **csharp - C# Script** in the Polyglot cell kernel selector. Keep the VS Code cell language set to **Polyglot Notebook**, not plain C# or Python.

In [ ]:
#nullable enable
using System;
using System.IO;
using System.Text.RegularExpressions;
using System.Collections.Generic;
using System.Diagnostics;
using System.Linq;
using System.Net;
using System.Net.Http;
using System.Net.Http.Headers;
using System.Text;
using System.Text.Json;
using System.Text.Json.Nodes;
using System.Threading;
using System.Threading.Tasks;

### 1b. Define the settings helper

Run the complete definition below before calling `SharePointSettings.Load`.

In [ ]:
#nullable enable
public sealed record SharePointSettings(string Root, string Tenant, string SearchEndpoint, string SiteUrl,
    string Account, string Deployment, string Model, string Prefix, string AppClientId, string SearchMiClientId)
{
    public const string ApiVersion = "2026-08-01-preview";
    public const int Dimensions = 1536;
    public string Index => Prefix + "-nb7-index";
    public string DataSource => Prefix + "-nb7-ds";
    public string Skills => Prefix + "-nb7-skillset";
    public string Indexer => Prefix + "-nb7-indexer";
    public string FoundryEndpoint => "https://" + Account + ".services.ai.azure.com";
    public string EmbeddingEndpoint => "https://" + Account + ".openai.azure.com";
    public string StatePath => Path.Combine(Root, ".workshop", "lab07-" + Prefix + ".json");

    public static SharePointSettings Load(string root)
    {
        LoadEnvFile(Path.Combine(root, ".env"));
        LoadEnvFile(Path.Combine(root, ".env.azure"));
        string Required(string name) => Environment.GetEnvironmentVariable(name)?.Trim() is { Length: > 0 } value
            ? value : throw new InvalidOperationException("Missing setting: " + name + ". See .env.example.");
        string Optional(string name) => Environment.GetEnvironmentVariable(name)?.Trim() ?? "";
        var settings = new SharePointSettings(Path.GetFullPath(root), Required("AZURE_TENANT_ID"),
            Required("AZURE_SEARCH_ENDPOINT").TrimEnd('/'), Required("SHAREPOINT_SITE_URL").TrimEnd('/'),
            Required("AZURE_AI_ACCOUNT_NAME"), Required("AZURE_AI_EMBEDDING_DEPLOYMENT_NAME"),
            Optional("AZURE_AI_EMBEDDING_MODEL_NAME") is { Length: > 0 } model ? model : "text-embedding-3-small",
            Required("WORKSHOP_PREFIX"), Optional("SHAREPOINT_APP_CLIENT_ID"), Optional("SHAREPOINT_SEARCH_MI_CLIENT_ID"));
        settings.Validate();
        return settings;
    }

    public void Validate()
    {
        if (!Guid.TryParseExact(Tenant, "D", out _) || !Regex.IsMatch(Prefix, @"\A[a-z][a-z0-9]{2,19}\z"))
            throw new InvalidOperationException("Use a tenant GUID and a 3-20 lowercase letter/digit prefix starting with a letter.");
        if (!Regex.IsMatch(SearchEndpoint, @"\Ahttps://[a-z0-9][a-z0-9-]*\.search\.windows\.net\z"))
            throw new InvalidOperationException("Use a public-cloud HTTPS Search endpoint without path, query or credentials.");
        if (!Uri.TryCreate(SiteUrl, UriKind.Absolute, out var site) || site.Scheme != "https" ||
            !site.Host.EndsWith(".sharepoint.com", StringComparison.OrdinalIgnoreCase) || !site.IsDefaultPort ||
            site.UserInfo != "" || site.Query != "" || site.Fragment != "" ||
            SiteUrl.IndexOfAny(new[] { ';', '"', '\r', '\n', '\\' }) >= 0)
            throw new InvalidOperationException("Use a public-cloud SharePoint HTTPS site URL, not a file or sharing-link URL.");
        foreach (string value in new[] { Account, Deployment, Model })
            if (!Regex.IsMatch(value, @"\A[a-zA-Z0-9][a-zA-Z0-9_.-]{0,127}\z"))
                throw new InvalidOperationException("Use valid Foundry account, embedding deployment and model names.");
        if (!Regex.IsMatch(Account, @"\A[a-zA-Z0-9][a-zA-Z0-9-]{0,62}\z"))
            throw new InvalidOperationException("Use a Foundry account name, not an endpoint.");
    }

    public string ConnectionString()
    {
        if (!Guid.TryParseExact(AppClientId, "D", out var app) || !Guid.TryParseExact(SearchMiClientId, "D", out var mi))
            throw new InvalidOperationException("Set SHAREPOINT_APP_CLIENT_ID and SHAREPOINT_SEARCH_MI_CLIENT_ID to client/application GUIDs after the existing setup script completes.");
        if (app == mi) throw new InvalidOperationException("The ingestion app and Search managed identity must have different client IDs.");
        return $"SharePointOnlineEndpoint={SiteUrl};ApplicationId={app};FederatedCredentialApplicationId={mi};TenantId={Tenant}";
    }

    public static void ConfirmSetup(bool confirmed)
    {
        if (!confirmed)
            throw new InvalidOperationException("Review app consent, selected-site Read grant and Search identity federation, then explicitly use --setup-confirmed. This acknowledges review, not an automated permissions test.");
    }

    public static void LoadEnvFile(string path)
    {
        if (!File.Exists(path)) return;
        foreach (string raw in File.ReadLines(path))
        {
            string line = raw.Trim();
            if (line.StartsWith("export ", StringComparison.Ordinal)) line = line.Substring(7).TrimStart();
            if (line.Length == 0 || line.StartsWith("#", StringComparison.Ordinal)) continue;
            int equals = line.IndexOf('=');
            if (equals <= 0) continue;
            string name = line.Substring(0, equals).Trim(), value = line.Substring(equals + 1).Trim();
            if (!Regex.IsMatch(name, @"\A[A-Za-z_][A-Za-z0-9_]*\z")) continue;
            if (value.Length >= 2 && ((value[0] == '"' && value[value.Length - 1] == '"') ||
                (value[0] == '\'' && value[value.Length - 1] == '\'')))
                value = value.Substring(1, value.Length - 2);
            Environment.SetEnvironmentVariable(name, value);
        }
    }
}

### 1c. Define the REST helper

This defines the transport; it does not send requests or acquire tokens.

In [ ]:
#nullable enable
public sealed class SharePointTransport(HttpClient client, SharePointSettings settings,
    Func<string, Task<string>>? tokenProvider = null)
{
    public const string Audience = "https://search.azure.com";
    readonly Func<string, Task<string>> getToken = tokenProvider ?? (resource => AzureCliTokenAsync(settings.Tenant, resource));
    readonly HashSet<string> knownTokens = new(StringComparer.Ordinal);

    public async Task<JsonObject?> RequestAsync(HttpMethod method, string path, JsonObject? body = null,
        bool asUser = false, bool allowMissing = false)
    {
        string token = await getToken(Audience);
        if (string.IsNullOrWhiteSpace(token) || token.Any(char.IsControl))
            throw new InvalidOperationException("Azure CLI returned no usable Search token.");
        knownTokens.Add(token);
        if (asUser) RequireDelegatedUser(token, settings.Tenant);
        using var request = new HttpRequestMessage(method, settings.SearchEndpoint + "/" + path +
            "?api-version=" + SharePointSettings.ApiVersion);
        request.Headers.Authorization = new AuthenticationHeaderValue("Bearer", token);
        // Both headers identify the same local CLI user; no app-only or elevated-read substitution.
        if (asUser) request.Headers.Add("x-ms-query-source-authorization", token);
        if (body != null)
            request.Content = new StringContent(body.ToJsonString(), Encoding.UTF8, "application/json");
        using var response = await client.SendAsync(request);
        string text = Redact(await response.Content.ReadAsStringAsync());
        int status = (int)response.StatusCode;
        if (allowMissing && status == 404) return null;
        bool accepted = method == HttpMethod.Get ? status == 200 :
            method == HttpMethod.Put ? status is 200 or 201 or 204 :
            method == HttpMethod.Delete ? status is 200 or 204 :
            path.EndsWith("/docs/search", StringComparison.Ordinal) ? status == 200 : status is 200 or 202 or 204;
        if (!accepted)
        {
            string message = $"HTTP {status} {response.ReasonPhrase}: {method} {path}.";
            Console.Error.WriteLine(message);
            if (response.Headers.Date != null) Console.Error.WriteLine("Date (UTC): " + response.Headers.Date.Value.UtcDateTime.ToString("O"));
            foreach (string header in new[] { "x-ms-request-id", "request-id", "x-ms-correlation-request-id", "retry-after" })
                if (response.Headers.TryGetValues(header, out var values))
                    Console.Error.WriteLine(header + ": " + Redact(string.Join(", ", values)));
            Console.Error.WriteLine("Response body: " + (text.Length == 0 ? "(empty)" : text.Substring(0, Math.Min(2000, text.Length))));
            throw new HttpRequestException(message, null, response.StatusCode);
        }
        if (status is 202 or 204) return new JsonObject();
        try
        {
            return JsonNode.Parse(text)?.AsObject() ?? throw new JsonException();
        }
        catch (Exception error) when (error is JsonException or InvalidOperationException)
        {
            throw new InvalidOperationException($"HTTP {status}: {method} {path} returned invalid JSON.");
        }
    }

    public string Redact(string text)
    {
        foreach (string token in knownTokens) text = text.Replace(token, "[REDACTED]", StringComparison.Ordinal);
        return Regex.Replace(text, @"(?i)\bBearer\s+[^\s""<>]+", "Bearer [REDACTED]");
    }

    public static void RequireDelegatedUser(string token, string tenant)
    {
        try
        {
            string[] pieces = token.Split('.');
            if (pieces.Length != 3) throw new FormatException();
            string payload = pieces[1].Replace('-', '+').Replace('_', '/');
            payload = payload.PadRight((payload.Length + 3) / 4 * 4, '=');
            JsonObject claims = JsonNode.Parse(Encoding.UTF8.GetString(Convert.FromBase64String(payload)))!.AsObject();
            if (claims["idtyp"]?.ToString() == "app" || string.IsNullOrWhiteSpace(claims["scp"]?.ToString()) ||
                !Guid.TryParse(claims["oid"]?.ToString(), out _) ||
                !Guid.TryParse(claims["tid"]?.ToString(), out var actualTenant) ||
                actualTenant != Guid.Parse(tenant) ||
                claims["aud"]?.ToString().TrimEnd('/') != Audience)
                throw new FormatException();
        }
        catch (Exception error) when (error is FormatException or JsonException or InvalidOperationException or NullReferenceException)
        {
            throw new InvalidOperationException("Query requires a delegated Search token for a signed-in member user in AZURE_TENANT_ID. Sign out/in with Azure CLI; do not use an application-only token.");
        }
        // This is a local claim sanity check, not signature validation; Search enforces authorization.
    }

    public static async Task<string> AzureCliTokenAsync(string tenant, string resource)
    {
        if (!Guid.TryParseExact(tenant, "D", out _) || resource != Audience)
            throw new InvalidOperationException("Invalid Azure CLI tenant or token audience.");
        var start = new ProcessStartInfo
        {
            FileName = OperatingSystem.IsWindows() ? Environment.GetEnvironmentVariable("ComSpec") ?? "cmd.exe" : "az",
            UseShellExecute = false,
            RedirectStandardOutput = true,
            RedirectStandardError = true,
            CreateNoWindow = true,
        };
        if (OperatingSystem.IsWindows())
            foreach (string argument in new[] { "/d", "/c", "az" }) start.ArgumentList.Add(argument);
        foreach (string argument in new[] { "account", "get-access-token", "--resource", resource, "--tenant", tenant,
            "--query", "accessToken", "--output", "tsv", "--only-show-errors" })
            start.ArgumentList.Add(argument);
        using var process = Process.Start(start) ?? throw new InvalidOperationException("Could not start Azure CLI.");
        Task<string> output = process.StandardOutput.ReadToEndAsync(), errors = process.StandardError.ReadToEndAsync();
        using var timeout = new CancellationTokenSource(TimeSpan.FromSeconds(60));
        try { await process.WaitForExitAsync(timeout.Token); }
        catch (OperationCanceledException)
        {
            if (!process.HasExited) process.Kill(entireProcessTree: true);
            await process.WaitForExitAsync();
            await Task.WhenAll(output, errors);
            throw new InvalidOperationException("Azure CLI authentication timed out. Sign in to the workshop tenant.");
        }
        await errors;
        if (process.ExitCode != 0)
            throw new InvalidOperationException("Azure CLI authentication failed. Run az login --tenant with AZURE_TENANT_ID.");
        return (await output).Trim();
    }
}

### 1d. Load configuration and initialize the client

Run this only after cells **1a-1c** succeed. It reads local settings but makes no network requests.

In [ ]:
#nullable enable

string root = Path.GetFullPath("..");
SharePointSettings settings = SharePointSettings.Load(root);
HttpClient client = new HttpClient { Timeout = TimeSpan.FromSeconds(100) };
Func<string, Task<string>> tokenProvider = resource => SharePointTransport.AzureCliTokenAsync(settings.Tenant, resource);
Func<TimeSpan, Task> pause = duration => Task.Delay(duration);
SharePointTransport transport = new SharePointTransport(client, settings, resource => tokenProvider(resource));
string? previous_run_start = null;
Console.WriteLine("Search: " + settings.SearchEndpoint);
Console.WriteLine("SharePoint site: " + settings.SiteUrl);
Console.WriteLine("Index: " + settings.Index + " | API: " + SharePointSettings.ApiVersion);
Console.WriteLine("Foundry enrichment: " + settings.FoundryEndpoint);


## 2. Register and authorize the SharePoint ingestion app

`scripts\setup-sharepoint-app.ps1` performs 2a, 2b, the site grant in 2c when `-SiteUrl` is provided, and optional consent in one run; the steps below explain each operation. Complete setup before creating the SharePoint data source. One configured ingestion app can be reused by multiple indexes on this Search service.

### 2a. Reuse the configured application and understand its IDs

After running `scripts\setup-sharepoint-app.ps1`, reuse the existing ingestion app, such as `iq-workshop-sharepoint-indexer`. Its **Application (client) ID** is the `SHAREPOINT_APP_CLIENT_ID` setting in `.env`. The registration's separate **Object ID** is used for Graph app-management operations, not the notebook's ingestion connection. The app is **single tenant**; this secretless ingestion flow needs no redirect URI, no public-client flow and no client secret.

App creation and configuration are handled by the PowerShell script, not by notebook code. Reuse that app and review its permissions, federation and site grant below; do not create a duplicate app registration.

Use the **App registration permissions** table at the start of this notebook. This Entra ACL path needs the two Microsoft Graph application permissions, not SharePoint API permissions for native-group resolution. The shared setup script also configures native-group permissions for other scenarios; an existing app with that superset can be reused without changing its grants.

**Scope warning:** `Sites.Selected` requires an explicit grant on each workshop site; consent alone is insufficient. However, the separately required **Graph `Files.Read.All` is tenant-wide file-read permission**. This documented combination is **not strict site-only isolation** of the app. Review this with the tenant administrator and use a dedicated test tenant/site. We do not request tenant-wide `Sites.FullControl.All`. Do not silently remove a required permission or claim that the site URL in the data source limits the app's underlying authorization.

In the portal, verify these permissions under their **correct API surfaces**, with type **Application** and admin consent granted. If consent is missing, have an authorized administrator choose **Grant admin consent**. Requested permissions and consented permissions are different states.

<img src="../docs/sharepoint-ingestion-app-permissions.png" alt="Ingestion app registration API permissions showing application permissions and granted admin consent" width="1000" />

In **App registrations -> your ingestion app -> API permissions**, check the **API**, **Type** and **Status** columns. This example shows Microsoft Graph `Files.Read.All` and `Sites.Selected` as **Application** permissions with admin consent granted. It also shows SharePoint API permissions from the broader shared setup; those additional permissions are not required for this notebook's Entra-only ACL path. The screenshot is an example, not a request to copy every grant. Admin consent does not replace the explicit selected-site grant in section 2c.

### 2b. Understand the configured federation (no client secret)

The PowerShell setup script has already configured the ingestion app's federated credential to trust **this Azure AI Search service's system-assigned managed identity**. No additional credential or client secret needs to be created in this notebook. The screenshots below explain the existing configuration and how its identities relate.

**What federation means:** the ingestion app trusts a token from this specific Azure AI Search managed identity as proof that Search can authenticate as the app. Entra can exchange that proof for an access token for the ingestion app, without storing a client secret in Search. The app's consented API permissions and site grant determine what ingestion may read; federation itself grants neither permission nor site access. It does not change which signed-in users can query restricted documents.

<img src="../docs/sharepoint-ingestion-federated-credentials.png" alt="Ingestion app Certificates and secrets page with a federated credential named search-managed-identity" width="1000" />

On the **ingestion app registration**, the **Federated credentials** tab records this trust. The example has a credential named `search-managed-identity` and no client secrets. The name is only a label: inspect the credential to verify its subject, issuer and audience. The subject is redacted in this screenshot, so the image alone does not prove that the correct identity is trusted.

<img src="../docs/sharepoint-ingestion-search-managed-identity.png" alt="Azure AI Search Identity page showing the system-assigned managed identity enabled and its redacted Object principal ID" width="1000" />

The trusted identity comes from **your Azure AI Search resource -> Identity -> System assigned**, with status **On**. Its **Object (principal) ID** must match the federated credential's **subject**. This is Search's identity, not the ingestion app's object ID or a Foundry identity. The credential uses your tenant's Entra issuer and the `api://AzureADTokenExchange` audience. The separate Search identity **Application/client ID** is what you save as `SHAREPOINT_SEARCH_MI_CLIENT_ID`; do not substitute the principal ID.

| Identifier | Used for |
|---|---|
| Ingestion app client ID | `ApplicationId` in the data source connection string |
| Search managed identity client/application ID | `FederatedCredentialApplicationId` in the data source connection string |
| Search managed identity principal/object ID | Subject trusted by the app's federated credential |
| Ingestion app object ID | Graph `/applications/{id}` operations only |
| Federated credential object's own ID | Not used in the data source connection string |

### 2c. Grant the app access to the selected test site

An administrator must make an explicit site permission grant **in addition to Entra admin consent**. The setup script grants site-scoped **Read** on the dedicated test site. The Search ACL permission matrix documents `Sites.Selected` and an explicit site grant, but does not specify the minimum site-grant role for every ACL scenario. Verify ACL extraction and two-user query behavior; content-ingestion success alone does not prove ACL support. Do not automatically escalate to FullControl or substitute a production site.

Run the setup script in a **separate PowerShell terminal** from the repository root, using the ingestion app's display name and your site URL:

```powershell
.\scripts\setup-sharepoint-app.ps1 -AppName 'iq-workshop-sharepoint-indexer' `
  -SiteUrl 'https://<tenant>.sharepoint.com/sites/<workshop-site>'
```

The script grants and verifies site-scoped Read directly through Microsoft Graph. Existing grants that allow reading are reused; broader grants are left unchanged with a warning. No additional app registration or PnP setup is needed. By default, the signed-in caller needs Graph delegated `Sites.FullControl.All`, a SharePoint Administrator (or higher) role, and target-site membership; unauthorized preflight stops before changing the app.

If the CLI token cannot manage site grants, an authorized **Privileged Role Administrator or Global Administrator** can explicitly opt in to app-only bootstrap instead:

```powershell
.\scripts\setup-sharepoint-app.ps1 -AppName 'iq-workshop-sharepoint-indexer' `
  -SiteUrl 'https://<tenant>.sharepoint.com/sites/<workshop-site>' `
  -BootstrapSiteAccess -GrantAdminConsent
```

Bootstrap requires Graph authorization to manage credentials and app-role assignments. It temporarily consents tenant-wide Graph `Sites.FullControl.All` on the ingestion app's service principal and adds a 15-minute credential to grant the site **Read**, then attempts and verifies cleanup in `finally`, including on failure. No runtime secret is added, and pre-existing credentials/permissions are preserved. Tokens/credentials are not printed or written to `.env`; issued tokens can remain valid until expiry. Forced termination can prevent cleanup; any reported cleanup failure requires manual action. This option does not bypass administrator authorization. Runtime admin consent remains separate/opt-in (`-GrantAdminConsent`).

## 3. Indexed path: create the SharePoint data source

After completing the registration, consent, federation and site grant, set `INDEXED_SETUP_CONFIRMED = true`. This confirms that setup was reviewed; it is **not an automated permissions test**. A successful indexer run and two-user permission exercise are still required.

The connection string contains only non-secret IDs. Reuse the existing app and federation rather than introducing a client secret. This authenticates ingestion; the query token later identifies the user whose document permissions Search evaluates.

`defaultSiteLibrary` scopes this lab to the selected site's default document library. It does not crawl all tenant sites, lists, or site pages. All files in that test library can be indexed; use a dedicated library/site containing only workshop content.

In [ ]:
bool INDEXED_SETUP_CONFIRMED = true;
if (!INDEXED_SETUP_CONFIRMED)
    throw new InvalidOperationException("Complete section 2 before indexed ingestion.");
string connection_string = settings.ConnectionString();
Console.WriteLine("Indexed setup acknowledged. The connection uses distinct client IDs, not a secret.");
Console.WriteLine("This is not an automated consent, federation or site-access test.");

In [ ]:
JsonObject data_source = new JsonObject
{
    ["name"] = settings.DataSource, ["type"] = "sharepoint",
    ["credentials"] = new JsonObject { ["connectionString"] = connection_string },
    ["container"] = new JsonObject { ["name"] = "defaultSiteLibrary" },
    ["indexerPermissionOptions"] = new JsonArray("userIds", "groupIds"),
};
await transport.RequestAsync(HttpMethod.Put, $"datasources/{settings.DataSource}", data_source);
Console.WriteLine("SharePoint data source ready: " + settings.DataSource);

### Index schema: text vectors, source metadata and Entra ACLs

`permissionFilterOption='enabled'` makes Search match the querying user's Entra identity against `UserIds` and `GroupIds`. The ACL fields are filterable but not retrievable in normal results. Like the reference notebook, this index does not configure native SharePoint-group resolution.

`content` holds Markdown, `contentVector` holds its embedding, and `sourceFile`, `webUrl`, `page` and `pageTo` identify the evidence. SharePoint ACL ingestion also requires the exact `metadata_spo_item_path` field; a friendly web URL does not replace it. Project the source path and both ACL collections onto every chunk.

The embedding skill and query-time Azure OpenAI vectorizer use the same deployment, model and 1536 dimensions. The initial wildcard query deliberately checks visibility before testing relevance.

In [ ]:
JsonObject index = JsonNode.Parse("""
    {
      "permissionFilterOption": "enabled",
      "fields": [
        {"name":"id","type":"Edm.String","key":true,"searchable":true,"analyzer":"keyword"},
        {"name":"parent_id","type":"Edm.String","filterable":true},
        {"name":"content","type":"Edm.String","searchable":true},
        {"name":"contentVector","type":"Collection(Edm.Single)","searchable":true,"retrievable":false,"dimensions":1536,"vectorSearchProfile":"text-profile"},
        {"name":"sourceFile","type":"Edm.String","searchable":true,"filterable":true},
        {"name":"webUrl","type":"Edm.String"},
        {"name":"page","type":"Edm.Int32","filterable":true},
        {"name":"pageTo","type":"Edm.Int32","filterable":true},
        {"name":"metadata_spo_item_id","type":"Edm.String","filterable":true},
        {"name":"metadata_spo_item_path","type":"Edm.String","searchable":false,"filterable":true,"retrievable":false},
        {"name":"lastModified","type":"Edm.DateTimeOffset","filterable":true},
        {"name":"UserIds","type":"Collection(Edm.String)","permissionFilter":"userIds","filterable":true,"retrievable":false},
        {"name":"GroupIds","type":"Collection(Edm.String)","permissionFilter":"groupIds","filterable":true,"retrievable":false}
      ],
      "vectorSearch": {
        "algorithms":[{"name":"hnsw","kind":"hnsw","hnswParameters":{"metric":"cosine"}}],
        "profiles":[{"name":"text-profile","algorithm":"hnsw","vectorizer":"aoai-vectorizer"}],
        "vectorizers":[{"name":"aoai-vectorizer","kind":"azureOpenAI","azureOpenAIParameters":{}}]
      },
      "semantic": {
        "defaultConfiguration":"semantic-config",
        "configurations":[{"name":"semantic-config","prioritizedFields":{
          "titleField":{"fieldName":"sourceFile"},
          "prioritizedContentFields":[{"fieldName":"content"}]
        }}]
      }
    }
    """)!.AsObject();
index["name"] = settings.Index;
index["vectorSearch"]!["vectorizers"]![0]!["azureOpenAIParameters"] = new JsonObject
{
    ["resourceUri"] = settings.EmbeddingEndpoint,
    ["deploymentId"] = settings.Deployment,
    ["modelName"] = settings.Model,
};
await transport.RequestAsync(HttpMethod.Put, $"indexes/{settings.Index}", index);
Console.WriteLine("Index ready: " + settings.Index);

## 4. Content Understanding, text embeddings and ACL projections

Content Understanding reads the original file and emits Markdown `text_sections` using the reference notebook's semantic chunking: up to 800 tokens per section, with no overlap. `locationMetadata` supplies page ranges. Each section is embedded with the workshop text-embedding deployment; no Text Split or image-processing skill is used.

**Every chunk must inherit the source ACLs.** The projection copies `/document/metadata_user_ids` and `/document/metadata_group_ids` into the permission-filter fields. With `skipIndexingParentDocuments`, parent-only indexer field mappings are insufficient. We avoid knowledge stores, enrichment caches, debug sessions and custom Web API skills, which have SharePoint ACL preview limitations.

In [ ]:
JsonObject skillset = JsonNode.Parse("""
    {
      "description":"Content Understanding semantic text sections, embeddings and Entra ACLs.",
      "cognitiveServices":{"@odata.type":"#Microsoft.Azure.Search.AIServicesByIdentity"},
      "skills":[
        {
          "@odata.type":"#Microsoft.Skills.Util.ContentUnderstandingSkill",
          "name":"content-understanding","context":"/document",
          "extractionOptions":["locationMetadata"],
          "chunkingProperties":{"method":"semantic","unit":"tokens","maximumLength":800,"overlapLength":0},
          "inputs":[{"name":"file_data","source":"/document/file_data"}],
          "outputs":[{"name":"text_sections","targetName":"text_sections"}]
        },
        {
          "@odata.type":"#Microsoft.Skills.Text.AzureOpenAIEmbeddingSkill",
          "name":"text-embed","context":"/document/text_sections/*","dimensions":1536,
          "inputs":[{"name":"text","source":"/document/text_sections/*/content"}],
          "outputs":[{"name":"embedding","targetName":"content_vector"}]
        }
      ],
      "indexProjections":{
        "selectors":[{
          "parentKeyFieldName":"parent_id","sourceContext":"/document/text_sections/*",
          "mappings":[
            {"name":"content","source":"/document/text_sections/*/content"},
            {"name":"contentVector","source":"/document/text_sections/*/content_vector"},
            {"name":"page","source":"/document/text_sections/*/locationMetadata/pageNumberFrom"},
            {"name":"pageTo","source":"/document/text_sections/*/locationMetadata/pageNumberTo"},
            {"name":"sourceFile","source":"/document/metadata_spo_item_name"},
            {"name":"metadata_spo_item_id","source":"/document/metadata_spo_item_id"},
            {"name":"webUrl","source":"/document/metadata_spo_item_weburi"},
            {"name":"metadata_spo_item_path","source":"/document/metadata_spo_item_path"},
            {"name":"lastModified","source":"/document/metadata_spo_item_last_modified"},
            {"name":"UserIds","source":"/document/metadata_user_ids"},
            {"name":"GroupIds","source":"/document/metadata_group_ids"}
          ]
        }],
        "parameters":{"projectionMode":"skipIndexingParentDocuments"}
      }
    }
    """)!.AsObject();
skillset["name"] = settings.Skills;
skillset["cognitiveServices"]!["subdomainUrl"] = settings.FoundryEndpoint;
JsonObject embedding = skillset["skills"]![1]!.AsObject();
embedding["resourceUri"] = settings.EmbeddingEndpoint;
embedding["deploymentId"] = settings.Deployment;
embedding["modelName"] = settings.Model;
skillset["indexProjections"]!["selectors"]![0]!["targetIndexName"] = settings.Index;
await transport.RequestAsync(HttpMethod.Put, $"skillsets/{settings.Skills}", skillset);
Console.WriteLine("Skills ready: " + settings.Skills);

## 5. Create and monitor the SharePoint indexer

The first run starts on creation. Updating an existing idle indexer in this cell explicitly requests a new run; wait if it is already running. The SharePoint preview registration, application permissions, admin consent, selected-site grant, federated credential and Search permissions must all be correct. A successful data-source PUT alone does not prove that ingestion can read the site. No schedule is set; reruns are explicit.

This cell only submits ingestion, without monitoring or querying. It captures the previous run start
in `previous_run_start`; the monitor rejects that stale completed result while waiting for the new run.
Creation starts billable analysis/embeddings. Complete indexed configuration/source creation first.

In [ ]:
JsonObject? existing_status = await transport.RequestAsync(HttpMethod.Get,
    $"indexers/{settings.Indexer}/status", allowMissing: true);
bool indexer_exists = existing_status != null;
previous_run_start = null;
if (indexer_exists)
{
    JsonObject previous_run = existing_status!["lastResult"]?.AsObject() ?? new JsonObject();
    if (previous_run["status"]?.ToString() == "inProgress")
        throw new InvalidOperationException("The indexer is already running. Wait before updating it.");
    previous_run_start = previous_run["startTime"]?.ToString();
}
JsonObject indexer = new JsonObject
{
    ["name"] = settings.Indexer, ["dataSourceName"] = settings.DataSource,
    ["targetIndexName"] = settings.Index, ["skillsetName"] = settings.Skills,
    ["parameters"] = new JsonObject
    {
        ["batchSize"] = 1, ["maxFailedItems"] = 0, ["maxFailedItemsPerBatch"] = 0,
        ["configuration"] = new JsonObject
        { ["dataToExtract"] = "contentAndMetadata", ["allowSkillsetToReadFileData"] = true },
    },
};
await transport.RequestAsync(HttpMethod.Put, $"indexers/{settings.Indexer}", indexer);
Console.WriteLine("SharePoint indexer ready: " + settings.Indexer);
if (indexer_exists)
{
    await transport.RequestAsync(HttpMethod.Post, $"indexers/{settings.Indexer}/run");
    Console.WriteLine("New run requested for the existing indexer.");
}

In [ ]:
bool indexer_completed = false;
JsonObject indexer_status = new JsonObject();
for (int attempt = 0; attempt < 90; attempt++)
{
    indexer_status = (await transport.RequestAsync(HttpMethod.Get, $"indexers/{settings.Indexer}/status"))!;
    JsonObject last_run = indexer_status["lastResult"]?.AsObject() ?? new JsonObject();
    string? state = last_run["status"]?.ToString();
    Console.WriteLine($"Run: {state} | processed: {last_run["itemsProcessed"]} | failed: {last_run["itemsFailed"]}");
    bool is_requested_run = previous_run_start == null || last_run["startTime"]?.ToString() != previous_run_start;
    if (state == "success" && is_requested_run)
    {
        Console.WriteLine("Warnings: " + (last_run["warnings"] ?? new JsonArray())
            .ToJsonString(new JsonSerializerOptions { WriteIndented = true }));
        if ((last_run["itemsFailed"]?.GetValue<int>() ?? 0) != 0 || (last_run["errors"]?.AsArray().Count ?? 0) != 0)
            throw new InvalidOperationException(last_run.ToJsonString(new JsonSerializerOptions { WriteIndented = true }));
        indexer_completed = true;
        break;
    }
    if ((is_requested_run && state is "transientFailure" or "persistentFailure" or "error") ||
        indexer_status["status"]?.ToString() == "error")
        throw new InvalidOperationException((last_run.Count == 0 ? indexer_status : last_run)
            .ToJsonString(new JsonSerializerOptions { WriteIndented = true }));
    if (attempt < 89) await pause(TimeSpan.FromSeconds(10));
}
if (!indexer_completed)
    throw new TimeoutException("Indexer did not complete within 15 minutes. Inspect status before retrying.");

## 6. Query indexed content as the signed-in user

`Authorization` authenticates access to Search. `x-ms-query-source-authorization` supplies the end-user identity for document permission filtering. In this local lab both represent your CLI-signed-in user, but their purposes are different. Do not substitute an application-only token for the end-user token.

Start with `question = '*'` to check document visibility independently of keyword matching. Then replace it with terms from your documents. The sample travel-policy query will not necessarily match an existing library's content.

`UserIds` contains Entra user object IDs; `GroupIds` contains the source's group permission entries. For this notebook, use Entra groups assigned directly to documents. Native SharePoint `spg:` entries and special owners claims are not ordinary Entra group GUIDs; this index does not expand them. A document can be visible in SharePoint through a different access path and still be filtered out here. See [supported group relationships](https://learn.microsoft.com/azure/search/search-indexer-sharepoint-access-control-lists#supported-group-relationships).

There is **no elevated-read header** in this lab. Search Index Data Contributor can perform elevated reads; do not use that privilege to claim that an ordinary user can access a document. For production query clients, use appropriately restricted Search roles.

An empty result is valid when no authorized content matches. It is not proof that the indexer failed or that all permissions are correct.

The C# transport uses one freshly acquired CLI user token for both headers, for their different purposes.
It locally rejects obvious application-only, wrong-tenant and wrong-audience token claims before querying;
this is **not signature validation or an authorization boundary**. Search validates/enforces the actual token.
Pretty JSON preserves content, source/page metadata and counts. It makes no extra query or follow-up request.

In [ ]:
string question = "*";
if (string.IsNullOrWhiteSpace(question)) throw new ArgumentException("Use a non-empty question or '*'.");
JsonObject indexed_result = (await transport.RequestAsync(HttpMethod.Post,
    $"indexes/{settings.Index}/docs/search",
    new JsonObject
    {
        ["search"] = question, ["count"] = true, ["top"] = 5,
        ["select"] = "sourceFile,webUrl,page,pageTo,content",
    },
    asUser: true))!;
Console.WriteLine("Visible chunks: " + indexed_result["@odata.count"]);
Console.WriteLine(indexed_result.ToJsonString(new JsonSerializerOptions { WriteIndented = true }));

### Prove permission trimming and refresh behavior

Use two **non-admin member accounts**, not guest users, and confirm their actual access by opening the restricted document in SharePoint. Give account A access directly, then repeat using an **Entra group assigned directly to the document**. Account B must not have access via another group, sharing link or site-admin privilege. Run the same query and index name for both accounts, signing out/in to Azure CLI and restarting the kernel between identities. Both need Search Index Data Reader; creating Search objects separately requires Search Service Contributor.

Expected: account A can retrieve the restricted document; account B cannot. Remove the direct user grant before the group-only check so that a `UserIds` match cannot hide a group-resolution failure. Record document URLs and observed results without saving source content into the committed notebook. The notebook never changes SharePoint permissions automatically, and a successful direct-user test does not prove group access.

Next change document permissions, then opt in to run the indexer again. Item-level unique ACL changes refresh on successful runs. **Inherited site/library/folder ACL changes need an explicit permissions resync** (or a supported document reset) and then an indexer run. The following cell demonstrates the resync; it touches only this named indexer. Rerun the monitor and query cells afterwards.

Changing a skillset instead requires reprocessing content, not just an ACL resync. Indexed content/permissions have synchronization latency; remote retrieval follows Microsoft 365's own indexing and permission-propagation behavior, not instantaneous document scanning.

In [ ]:
bool RUN_AGAIN = false;
bool RESYNC_PERMISSIONS = false;
if (RUN_AGAIN)
{
    JsonObject run_status = (await transport.RequestAsync(HttpMethod.Get, $"indexers/{settings.Indexer}/status"))!;
    JsonObject last_run = run_status["lastResult"]?.AsObject() ?? new JsonObject();
    if (last_run["status"]?.ToString() == "inProgress")
        throw new InvalidOperationException("The indexer is already running. Wait for it to finish.");
    previous_run_start = last_run["startTime"]?.ToString();
    if (RESYNC_PERMISSIONS)
        await transport.RequestAsync(HttpMethod.Post, $"indexers/{settings.Indexer}/resync",
            new JsonObject { ["options"] = new JsonArray("permissions") });
    await transport.RequestAsync(HttpMethod.Post, $"indexers/{settings.Indexer}/run");
    Console.WriteLine("Run requested. Rerun monitor-indexer, then query-index.");
}
else
    Console.WriteLine("No additional indexer run or ACL resync requested.");
// After requesting a run, rerun monitor-indexer, then query-index.

## 7. Optional Search-object cleanup and separate tenant cleanup

Enable the following cell to remove only this notebook's named Search objects. Missing objects (404) are explicitly allowed; authorization failures and other errors are not ignored.

**This does not remove the ingestion app, its enterprise application, federated credential, consent, selected-site grant or source files.** First check whether other indexes still use the app. When no longer needed, use an authorized SharePoint/Graph administration workflow to revoke the site grant and remove the federated trust/consented permissions or delete the dedicated test app and service principal. Remove only documents/site created for this exercise, not a pre-existing app or site. App registrations are tenant objects and are not deleted by deleting the Azure resource group.

Clear all notebook outputs before sharing; outputs can contain private SharePoint document text and URLs even though tokens are not printed.

In [ ]:
bool CLEAN_UP = false;
if (CLEAN_UP)
{
    foreach (var item in new[]
    {
        (Collection: "indexers", Name: settings.Indexer),
        (Collection: "skillsets", Name: settings.Skills),
        (Collection: "indexes", Name: settings.Index),
        (Collection: "datasources", Name: settings.DataSource),
    })
    {
        await transport.RequestAsync(HttpMethod.Delete, $"{item.Collection}/{item.Name}", allowMissing: true);
        Console.WriteLine("Deleted or already absent: " + item.Name);
    }
}
else
    Console.WriteLine("Search-object cleanup disabled. Tenant app/site cleanup remains a separate admin step.");

In [ ]:
client.Dispose();

## References and sample lineage

Adapted from [foundry-iq-sharepoint-indexer: 01_setup_index.ipynb](https://github.com/memasanz/foundry-iq-sharepoint-indexer/blob/main/notebooks/01_setup_index.ipynb). The text extraction, embeddings, source/page metadata, Entra ACL projections and REST query pattern are retained. The Vision vectorizer/skill, image projections and image descriptions are omitted. Workshop environment names and explicit CLI-user authentication are retained so the query identity is unambiguous.

- [SharePoint indexer and app registration](https://learn.microsoft.com/azure/search/search-how-to-index-sharepoint-online)
- [ACL permissions matrix, native SharePoint groups, identifier lookup and refresh](https://learn.microsoft.com/azure/search/search-indexer-sharepoint-access-control-lists)
- [Grant a site permission with Microsoft Graph](https://learn.microsoft.com/graph/api/site-post-permissions)
- [Query-time ACL enforcement](https://learn.microsoft.com/azure/search/search-query-access-control-rbac-enforcement)
- [Content Understanding skill and semantic chunking](https://learn.microsoft.com/azure/search/cognitive-search-skill-content-understanding)

Skill/API versions and preview access can change. This notebook does not promise complete SharePoint permission-model parity. For retrieval without an index, continue with notebook 08.